In [26]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F



In [27]:
tokenizor = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [28]:
text = "The capital city of france is [MASK]."

In [29]:
inputs = tokenizor(text, return_tensors="pt")

In [30]:
with torch.inference_mode():
    logits = model(**inputs).logits

In [31]:
mask_index = (inputs.input_ids == tokenizor.mask_token_id).nonzero(as_tuple=True)[1][0].item()


In [32]:
mask_logits = logits[0, mask_index, :]

In [33]:
probs = F.softmax(mask_logits, dim=-1)
top_k = torch.topk(probs, k=5)

In [34]:
print("Top 5 predictions for [MASK]:")
for token_id, prob in zip(top_k.indices.tolist(), top_k.values.tolist()):
    token = tokenizor.convert_ids_to_tokens([token_id])[0]  
    print(f"{token:<12} -> {prob:.4f}")

Top 5 predictions for [MASK]:
paris        -> 0.3465
lille        -> 0.0925
lyon         -> 0.0850
marseille    -> 0.0623
toulouse     -> 0.0405
